In [1]:
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import quote_plus

/disk1/anaconda3/envs/kcat/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
df = pd.read_csv('data/brenda_data.csv')
df.head()

,enzyme_name,ec_number,turnover_number,substrate,organism,uniprot,commentary
0,acetate kinase,2.7.2.1,880.0,formate,Salmonella enterica subsp. enterica serovar Ty...,P63411,"pH 7.5, 37°C"
1,acetate kinase,2.7.2.1,17.0,acetate,Methanosarcina thermophila,P38502,"mutant enzyme G331Q/I332M, at pH 5.5 and 45Â°C"
2,acetate kinase,2.7.2.1,80.0,acetate,Lactococcus lactis,A2RNG4,"pH 7.5, 30°C, recombinant His-tagged isozyme A..."
3,acetate kinase,2.7.2.1,195.0,acetate,Shewanella sp.,E3W769,"pH 7.5, 10°C, recombinant enzyme"
4,acetate kinase,2.7.2.1,233.0,acetate,Methylotuvimicrobium alcaliphilum,G4T0C7,"pH 8.0, 30°C"


In [ ]:
df = df.drop_duplicates()
print("After removing duplicates: ", len(df))

중복 제거 후 데이터:  34554


## Extract pH and Temperature

In [ ]:
def extract_pH(comment):
    if pd.isna(comment):
        return None
    parts = comment.split(',')
    for part in parts:
        if 'pH' in part:
            match = re.search(r'pH\s*([\d.]+)', part)
            if match:
                try:
                    return float(match.group(1))
                except:
                    return None
    return None

def extract_temperature(comment):
    if pd.isna(comment):
        return None
    parts = comment.split(',')
    for part in parts:
        if '°C' in part:
            match = re.search(r'([\d.]+)\s*°C', part)
            if match:
                try:
                    return float(match.group(1))
                except:
                    return None
    return None

In [5]:
df['pH'] = df['commentary'].apply(extract_pH)
df['temperature'] = df['commentary'].apply(extract_temperature)

In [ ]:
df = df.dropna(subset=['pH', 'temperature'])
print(f"Data with both pH and temperature: {len(df)}")

pH와 온도가 모두 있는 데이터 수: 25111


## Add wildtype/mutant type information

In [ ]:
def extract_type_and_mutation(comment):
    if pd.isna(comment):
        return None, None
    
    is_wildtype = any(word in comment.lower() for word in ['wildtype', 'wild-type', 'wild type'])
    is_mutant = 'mutant' in comment.lower()
    
    mutations = re.findall(r'([A-Z]\d+[A-Z])', comment)
    
    if is_wildtype:
        return 'wildtype', None
    elif is_mutant:
        if mutations:
            return 'mutant', '/'.join(mutations)
        else:
            return 'mutant', None
    else:
        return None, None

df['type'] = None
df['mutation'] = None

for idx, row in df.iterrows():
    type_val, mutation_val = extract_type_and_mutation(row['commentary'])
    df.at[idx, 'type'] = type_val
    df.at[idx, 'mutation'] = mutation_val

In [ ]:
no_type_info = df[df['type'].isna()]
print(f"Data with no type information: {len(no_type_info)}")

# If there is no type information, suppose it is a wildtype
df['type'] = df['type'].fillna('wildtype')

# If the data is a mutant but have no mutation information, remove it
df = df[~((df['type'] == 'mutant') & (df['mutation'].isna()))]

print("\nTotal data length: ", len(df))
print(f"Wildtype data length: {len(df[df['type'] == 'wildtype'])}")
print(f"Mutant data length: {len(df[df['type'] == 'mutant'])}")

타입 정보가 없는 데이터 수: 10533

총 데이터 수:  24729
Wildtype 데이터 수: 14435
Mutant 데이터 수: 10294


# Add sequence information

In [ ]:
df = df[~df['uniprot'].str.contains(',', na=False)]
print('Data with one Uniprot ID: ', len(df))

Uniprot 값이 한 개만 있는 데이터:  23978


In [ ]:
def get_protein_sequence(uniprot_id: str):
    base_url = "https://rest.uniprot.org/uniprotkb/"
    url = f"{base_url}{uniprot_id}.fasta"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise exception for HTTP errors
        
        # Parse the FASTA format
        lines = response.text.split('\n')
        if len(lines) > 1:
            # Join all lines except the first one (header)
            sequence = ''.join(lines[1:])
            return sequence
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching sequence for {uniprot_id}: {e}")
        return None

In [67]:
sequences = []

for i in tqdm(range(len(df))):
    try:
        seq = get_protein_sequence(df.iloc[i]['uniprot'])
        sequences.append(seq)
    except:
        sequences.append(None)

 11%|█         | 2579/23978 [46:48<6:26:11,  1.08s/it]

Error fetching sequence for ANI02794.1: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/ANI02794.1.fasta


 11%|█         | 2583/23978 [46:53<6:25:56,  1.08s/it]

Error fetching sequence for ANI02794.1: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/ANI02794.1.fasta


 19%|█▊        | 4447/23978 [1:20:37<5:52:53,  1.08s/it]

...

 98%|█████████▊| 23460/23978 [7:06:24<09:26,  1.09s/it]

Error fetching sequence for JN665070: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/JN665070.fasta


 98%|█████████▊| 23464/23978 [7:06:29<09:15,  1.08s/it]

Error fetching sequence for JN665070: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/JN665070.fasta


100%|██████████| 23978/23978 [7:15:52<00:00,  1.09s/it]


In [ ]:
# save data
df['sequence'] = sequences
df.to_csv('data/brenda_data_with_sequence.csv', index=False)

ValueError: Length of values (23625) does not match length of index (21822)

In [ ]:
len(df)

23978

In [12]:
df.isna().sum()

enzyme_name            0
ec_number              0
turnover_number        0
substrate              0
organism               0
uniprot                0
commentary             0
pH                     0
temperature            0
type                   0
mutation           13970
sequence             353
dtype: int64

In [ ]:
print("Uniprot extraction failed: ", len(df[df['sequence'].isna()]))
df = df.dropna(subset=['sequence'])
print("Data length": ", len(df))

Uniprot 추출 실패한 데이터:  353
데이터 길이:  23625


## Add substrate SMILES information

In [ ]:
substrate2_list = []
substrate3_list = []
drop_indices = []

exclude_words = ['with', 'as', 'cosubstrate', 'co-substrate:', 'cosubstrate:', 'co-substrate', '+', 'and']

for idx, row in df.iterrows():
    commentary = str(row['commentary'])
    
    if "cosubstrate" in commentary.lower() or "co-substrate" in commentary.lower():
        
        parts = commentary.split(', ')
        
        cosubstrate_parts = []
        for part in parts:
            if "cosubstrate" in part.lower() or "co-substrate" in part.lower():
                cosubstrate_parts.append(part)
        
        substrates = []
        for part in cosubstrate_parts:
            words = part.split(' ')
            filtered_words = [word for word in words 
                            if word.lower() not in [w.lower() for w in exclude_words]]
            substrates.extend(filtered_words)
        
        substrates = list(set([s for s in substrates if s.strip()]))
        
        if len(substrates) == 0:
            substrate2_list.append(None)
            substrate3_list.append(None)
        elif len(substrates) == 1:
            substrate2_list.append(substrates[0])
            substrate3_list.append(None)
        elif len(substrates) == 2:
            substrate2_list.append(substrates[0])
            substrate3_list.append(substrates[1])
        else: 
            drop_indices.append(idx)
            substrate2_list.append(None)
            substrate3_list.append(None)
    
    else:
        substrate2_list.append(None)
        substrate3_list.append(None)

df['substrate2'] = substrate2_list
df['substrate3'] = substrate3_list

In [ ]:
print(f"Original data length: {len(df)}")

df = df.drop(drop_indices)
print(f"Removed data length: {len(drop_indices)}")
print(f"Final data length: {len(df)}")

원본 데이터 개수: 23625
제거된 데이터 개수: 82
최종 데이터 개수: 23543


In [ ]:
for idx, row in df.iterrows():
    substrate = str(row['substrate'])
    substrate2 = row['substrate2']
    substrate3 = row['substrate3']
    
    if substrate2 is not None:
        df.at[idx, 'substrate'] = f"{substrate};{substrate2}"
    
    if substrate3 is not None:
        df.at[idx, 'substrate'] = f"{df.at[idx, 'substrate']};{substrate3}"

In [17]:
df.drop(['substrate2', 'substrate3'], axis=1, inplace=True)

In [ ]:
unique_substrates = set()

for substrate in df['substrate']:
    if ";" in str(substrate):
        split_substrates = [s.strip() for s in str(substrate).split(";")]
        unique_substrates.update(split_substrates)
    else:
        unique_substrates.add(str(substrate).strip())

unique_substrates_list = list(unique_substrates)

print("Unique substrates 개수:", len(unique_substrates_list))
print("Unique substrates:", unique_substrates_list)

Unique substrates 개수: 4459
Unique substrates: ['octanesulfonate', '4-anisyl alcohol', '2,5-dinitrophenyl-beta-D-xylopyranoside', 'YAYDTRYRR', 'Allantoate', 'vanillylamine', 'cholic acid', '3-O-alpha-D-glucopyranosyl-3-O-alpha-D-glucopyranosyl-3-O-alpha-D-glucopyranosyl-D-glucopyranose', 'nucleoside32 in EctRNAfMet2(CAU)', 'ADP-alpha-D-glucose', 'demethylsuberosin', 'phenethylamine', 'actin L-histidine73', 'GMP', '6,8-difluoromethylumbelliferyl phosphate', 'beta-D-Man-(1->4)-beta-D-Man-(1->4)-D-Man', 'amyloid beta-protein', '9,10-phenantrenequinone', 'N-terminal-L-methionyl-L-leucyl-glycyl-L-proline', 'FMN', 'factor Xa', 'Abz-GFSPFRSSRIGEIKEETT-Q-N-[2,4-dinitrophenyl]-ethylenediamine', '1-phosphate', 'poly(C)', 'HIV1 Gag protein N-terminal peptide', '3-O-methylgallate', '2-hydroxy-6-oxohepta-2,4-dienoate', 'GDP-beta-fucose', '4-nitrophenyl tetradecanoate', 'Urea', '(2Z)-2-(5-hydroxy-4,6-dimethyl-2-oxo-1,2-dihydro-3H-indol-3-ylidene)-N,N-di(prop-2-en-1-yl)hydrazinecarbothioamide', 'niger

In [ ]:
def get_cid_from_google(substrate):
    try:
        # PubChem PUG REST API URL
        search_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote_plus(substrate)}/cids/JSON"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(search_url, headers=headers)
        
        if response.status_code != 200:
            print(f"Failed to get response for {substrate}")
            return None
            
        data = response.json()
        if 'IdentifierList' in data and 'CID' in data['IdentifierList']:
            cids = data['IdentifierList']['CID']
            if cids:
                return str(cids[0])
                
        print(f"Could not find CID in PubChem for {substrate}")
        return None
        
    except Exception as e:
        print(f"Error getting CID from PubChem for {substrate}: {str(e)}")
        return None

def get_smiles(cid):
    try:
        # PubChem PUG REST API URL
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSMILES,CanonicalSMILES/JSON"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return None, None
            
        data = response.json()
        if 'PropertyTable' in data and 'Properties' in data['PropertyTable']:
            properties = data['PropertyTable']['Properties'][0]
            return properties.get('IsomericSMILES'), properties.get('CanonicalSMILES')
            
        return None, None
        
    except Exception as e:
        print(f"Error getting SMILES for CID {cid}: {str(e)}")
        return None, None

In [21]:
len(substrates)

4413

In [22]:
missing_in_substrates = set(unique_substrates_list) - set(substrates['substrate'])
missing_in_substrates = list(missing_in_substrates)

In [23]:
substrates.head()

,substrate,isomeric_smiles,canonical_smiles
0,formate,C(=O)[O-],C(=O)[O-]
1,acetate,CC(=O)[O-],CC(=O)[O-]
2,acetyl phosphate,NaN,NaN
3,ADP,C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H...,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(...
4,ATP,C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H...,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(...


In [ ]:
for substrate in missing_in_substrates:
    try:
        cid = get_cid_from_google(substrate)
        
        if cid is not None:
            isomeric_smiles, canonical_smiles = get_smiles(cid)

            if isomeric_smiles is not None:
                new_row = {
                    'substrate': substrate,
                    'isomeric_smiles': isomeric_smiles,
                    'canonical_smiles': canonical_smiles
                }
                substrates = pd.concat([substrates, pd.DataFrame([new_row])], ignore_index=True)
                print(f"추가됨: {substrate}")
    except:
        print(f"Error ({substrate}): {e}")

print(f"Final substrate length: {len(substrates)}")

Failed to get response for semialdehyde
Failed to get response for 4-estrene-3,17-dione
Failed to get response for acid
Failed to get response for 4'-demethyl-rebeccamycin
Failed to get response for geranyl
Failed to get response for 4-estrene-17beta-ol-3-one
Failed to get response for succinic
Failed to get response for 2-deoxy-D-erythrose
Failed to get response for 1
Failed to get response for using
Failed to get response for 5-phosphate
Failed to get response for protein]
Failed to get response for 1-phosphate
Failed to get response for cosubstrates
Failed to get response for D-2'-deoxycytidine
Failed to get response for tert-butyl
Failed to get response for crotyl
Failed to get response for tryparedoxin
Failed to get response for Q4
Failed to get response for geranylgeranyl
Failed to get response for cinnamyl
Failed to get response for malonyl-[acyl-carrier
Failed to get response for 4-phosphate
Failed to get response for Q
Failed to get response for viologen
Failed to get response

In [ ]:
def process_substrate_smiles(substrate):
    substrate_parts = [s.strip() for s in substrate.split(";")]
    
    isomeric_smiles = []
    canonical_smiles = []
    for part in substrate_parts:
        matching_row = substrates[substrates['substrate'] == part]
        
        if not matching_row.empty:
            isomeric = matching_row.iloc[0]['isomeric_smiles']
            canonical = matching_row.iloc[0]['canonical_smiles']
            
            if isomeric is not None:
                isomeric_smiles.append(isomeric)
                canonical_smiles.append(canonical)
        else:
            isomeric_smiles = []
            canonical_smiles = []

    try:
        isomeric_smiles = ";".join(isomeric_smiles)
        canonical_smiles = ";".join(canonical_smiles)
        return isomeric_smiles, canonical_smiles
    except:
        return None, None

In [27]:
isomeric_smiles, canonical_smiles = [], []

df = df.reset_index(drop=True)

for i in range(len(df)):
    isomeric, canonical = process_substrate_smiles(df.loc[i, 'substrate'])
    isomeric_smiles.append(isomeric)
    canonical_smiles.append(canonical)

df['isomeric_smiles'] = isomeric_smiles
df['canonical_smiles'] = canonical_smiles

In [ ]:
print('Data with no substrate SMILES: ', len(df[df['isomeric_smiles'].isna()]))

Substrate SMILES가 없는 데이터 수:  8271


## Add mutation information to sequence

In [ ]:
def apply_mutations(sequence, mutation_info):
    mutations = mutation_info.split('/')
    seq_len = len(sequence)

    for mutation in mutations:
        wt_aa = mutation[0]
        mt_aa = mutation[-1]
        pos = int(mutation[1:-1])-1 # Python number

        try:
            if sequence[pos] == wt_aa:
                sequence = sequence[:pos] + mt_aa + sequence[pos+1:]
            elif sequence[pos+1] == wt_aa: 
                sequence = sequence[:pos+1] + mt_aa + sequence[pos+2:]
            elif sequence[pos+2] == wt_aa:
                sequence = sequence[:pos+2] + mt_aa + sequence[pos+3:]
            else:
                print(f'Pos {pos} does not match wildtype amino acid {wt_aa}, {sequence[pos]}')
                print(f"Mutation {mutation} does not match wildtype amino acid {wt_aa} at position {pos}")
                print(f"Sequence: {sequence}")
                return None
        except:
            print(f"Mutation {mutation} is out of sequence length | sequence length: {seq_len}")
            return None

    if len(sequence) != seq_len:
        print(f"Sequence length changed from {seq_len} to {len(sequence)}")
        print(f"Sequence: {sequence}")
        return None
    
    return sequence

In [30]:
df = df.reset_index(drop=True)

In [31]:
df.head()

,enzyme_name,ec_number,turnover_number,substrate,organism,uniprot,commentary,pH,temperature,type,mutation,sequence,isomeric_smiles,canonical_smiles
0,acetate kinase,2.7.2.1,880.0,formate,Salmonella enterica subsp. enterica serovar Ty...,P63411,"pH 7.5, 37°C",7.5,37.0,wildtype,NaN,MSSKLVLVLNCGSSSLKFAIIDAVNGDEYLSGLAECFHLPEARIKW...,C(=O)[O-],C(=O)[O-]
1,acetate kinase,2.7.2.1,80.0,acetate,Lactococcus lactis,A2RNG4,"pH 7.5, 30°C, recombinant His-tagged isozyme A...",7.5,30.0,wildtype,NaN,MEKTLAVNAGSSSLKWQLYEMPEETVIAKGIFERIGLSGSISTTKF...,CC(=O)[O-],CC(=O)[O-]
2,acetate kinase,2.7.2.1,195.0,acetate,Shewanella sp.,E3W769,"pH 7.5, 10°C, recombinant enzyme",7.5,10.0,wildtype,NaN,MSDKLVLVLNCGSSSLKFAIIDAQSGDDKISGLAECFGLEDSRIKW...,CC(=O)[O-],CC(=O)[O-]
3,acetate kinase,2.7.2.1,233.0,acetate,Methylotuvimicrobium alcaliphilum,G4T0C7,"pH 8.0, 30°C",8.0,30.0,wildtype,NaN,MTIPNGNILVINSGSSSIKYRLIALPQEQVLADGLLERIGEQESRI...,CC(=O)[O-],CC(=O)[O-]
4,acetate kinase,2.7.2.1,233.3,acetate,Methylotuvimicrobium alcaliphilum,G4T0C7,"pH 8.0, 30°C",8.0,30.0,wildtype,NaN,MTIPNGNILVINSGSSSIKYRLIALPQEQVLADGLLERIGEQESRI...,CC(=O)[O-],CC(=O)[O-]


In [32]:
sequences = []

df = df.dropna(subset=['sequence']).reset_index(drop=True)

for i in range(len(df)):
    if pd.notna(df.loc[i, 'mutation']):
        seq = apply_mutations(df.iloc[i]['sequence'], df.iloc[i]['mutation'])
        sequences.append(seq)
    else:
        sequences.append(df.loc[i, 'sequence'])

Pos 65 does not match wildtype amino acid H, S
Mutation H66A does not match wildtype amino acid H at position 65
Sequence: MASPQLCRALVSAQWVAEALRAPRAGQPLQLLDASWYLPKLGRDARREFEERHIPGAAFFDIDQCSDRTSPYDHMLPGAEHFAEYAGRLGVGAATHVVIYDASDQGLYSAPRVWWMFRAFGHHAVSLLDGGLRHWLRQNLPLSSGKSQPAPAEFRAQLDPAFIKTYEDIKENLESRRFQVVDSRATGRFRGTEPEPRDGIEPGHIPGTVNIPFTDFLSQEGLEKSPEEIRHLFQEKKVDLSKPLVATCGSGVTACHVALGAYLCGKPDVPIYDGSWVEWYMRARPEDVISEGRGKTH
Mutation R440K is out of sequence length | sequence length: 229
Mutation R440H is out of sequence length | sequence length: 229
Mutation R440K is out of sequence length | sequence length: 229
Mutation R440H is out of sequence length | sequence length: 229
Mutation R440H is out of sequence length | sequence length: 229
Mutation R440K is out of sequence length | sequence length: 229
Mutation R440K is out of sequence length | sequence length: 229
Mutation R440H is out of sequence length | sequence length: 229
Mutation R440K is out of sequence length | sequence length: 229
Mut

In [33]:
df['sequence'] = sequences

In [ ]:
print('Sequence with unmatched mutation information: ', len(df[df['sequence'].isna()]))
df = df.dropna(subset=['sequence'])
print('Data length: ', len(df))

Sequence 정보가 틀린 데이터 수:  1801
df 길이:  21742


In [35]:
df.columns

Index(['enzyme_name', 'ec_number', 'turnover_number', 'substrate', 'organism',
       'uniprot', 'commentary', 'pH', 'temperature', 'type', 'mutation',
       'sequence', 'isomeric_smiles', 'canonical_smiles'],
      dtype='object')

In [ ]:
# save data
df.to_csv('data/brenda_sequence_substrates.csv', index=False)